In [1]:
import pandas as pd
import numpy as np

# Load clean data
df = pd.read_csv("../data/clean_articles.csv", encoding='utf-8-sig')

print("Data loaded!")
print("Shape:", df.shape)
print("\nSample:")
print(df['combined_text'][0][:200])

Data loaded!
Shape: (111860, 9)

Sample:
عالمی بینک عسکریت پسندی سے متاثرہ خاندانوں کی معاونت کرے گا اسلام باد عالمی بینک خیبرپختونخوا کے قبائلی اضلاع میں عسکریت پسندی سے پیدا ہونے والے بحران سے متاثرہ خاندانوں کی جلد بحالی بچوں کی صحت کی بہ


In [2]:
from sentence_transformers import SentenceTransformer
import torch

# Check if GPU is available (faster) or CPU (slower but works)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load multilingual BERT model
# This model understands Urdu, English and 100+ languages
print("\nLoading mBERT model... (first time downloads ~500MB, be patient)")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=device)

print("✅ Model loaded successfully!")
print("Model max sequence length:", model.max_seq_length)

c:\Users\User\anaconda3\envs\ultra_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu

Loading mBERT model... (first time downloads ~500MB, be patient)


c:\Users\User\anaconda3\envs\ultra_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6028.17it/s]


✅ Model loaded successfully!
Model max sequence length: 128


In [3]:
# Test on just 1 article before processing all 111k
sample_text = df['combined_text'][0]

print("Testing on sample article...")
print("Text:", sample_text[:100])

# Generate embedding for 1 article
embedding = model.encode(sample_text)

print("\n✅ Embedding generated!")
print("Embedding shape:", embedding.shape)
print("First 5 numbers:", embedding[:5])

Testing on sample article...
Text: عالمی بینک عسکریت پسندی سے متاثرہ خاندانوں کی معاونت کرے گا اسلام باد عالمی بینک خیبرپختونخوا کے قبا

✅ Embedding generated!
Embedding shape: (384,)
First 5 numbers: [ 0.0624284  -0.03750094 -0.15116169 -0.07263795  0.00453363]


In [4]:
import time

print("Generating embeddings for all 111,860 articles...")
print("This will take 30-60 minutes on CPU. Please wait...\n")

start_time = time.time()

# Process in batches of 64 for efficiency
embeddings = model.encode(
    df['combined_text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

end_time = time.time()
elapsed = (end_time - start_time) / 60

print(f"\n✅ All embeddings generated!")
print(f"Time taken: {elapsed:.1f} minutes")
print(f"Embeddings shape: {embeddings.shape}")

# Save embeddings immediately
np.save("../data/embeddings.npy", embeddings)
print("✅ Embeddings saved to data/embeddings.npy")

Generating embeddings for all 111,860 articles...
This will take 30-60 minutes on CPU. Please wait...



Batches: 100%|██████████| 1748/1748 [4:31:33<00:00,  9.32s/it]  



✅ All embeddings generated!
Time taken: 271.6 minutes
Embeddings shape: (111860, 384)
✅ Embeddings saved to data/embeddings.npy


In [5]:
# Verify embeddings saved correctly
embeddings = np.load("../data/embeddings.npy")

print("✅ Embeddings verified!")
print("Shape:", embeddings.shape)
print("Expected: (111860, 384)")
print("Total articles embedded:", embeddings.shape[0])
print("Embedding dimensions:", embeddings.shape[1])
print("File size: ~163 MB")

✅ Embeddings verified!
Shape: (111860, 384)
Expected: (111860, 384)
Total articles embedded: 111860
Embedding dimensions: 384
File size: ~163 MB
